# Find exact female-speaker matches in SPRINGLab IndicTTS-Hindi

This notebook compares the current Hindi Adult CI Auditory Training stimuli against the `SPRINGLab/IndicTTS-Hindi` transcript dataset and exports only exact full-utterance matches spoken by the female speaker. It does **not** cut individual words out of longer sentences, because doing so without alignment data can produce incorrect or clinically unsuitable clips.

**Licensing warning:** The dataset card says this corpus is governed by the original IIT Madras IndicTTS licence, not a simple permissive open-data licence. Review and obtain permission for your intended use before putting extracted audio in a public GitHub Pages site. This notebook writes extracted clips to the Colab session only; it does not upload them to GitHub.

Dataset: [SPRINGLab/IndicTTS-Hindi](https://huggingface.co/datasets/SPRINGLab/IndicTTS-Hindi) · Source/licence info: [IIT Madras IndicTTS database](https://www.iitm.ac.in/donlab/indictts/database)


In [ ]:
%pip install -q "datasets>=3.0.0" huggingface_hub soundfile pandas pyarrow


## 1. Load the app's current stimuli
The app JSON is fetched from the public GitHub repository. You can add or remove search phrases in the `EXTRA_TARGETS` list if you want to test more items. The match is based on exact normalized full-utterance text; punctuation/spacing differences are normalized conservatively.

In [ ]:
import json, re, unicodedata, hashlib, os, csv
from pathlib import Path
import pandas as pd
import requests

DATA_URL = 'https://raw.githubusercontent.com/noorain04-web/Hindi-Adult-CI-Auditory-Training/main/data/training-data.json'
response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()
app_data = response.json()

def normalize_text(value):
    if not isinstance(value, str): return ''
    value = unicodedata.normalize('NFC', value).strip()
    value = re.sub(r'[।.!?,;:]+$', '', value)
    value = re.sub(r'\s+', ' ', value)
    return value.casefold()

targets = {}
def add_target(s, origin):
    if isinstance(s, str) and s.strip():
        key = normalize_text(s)
        if key: targets.setdefault(key, {'text': s.strip(), 'origins': []})['origins'].append(origin)

for section, items in app_data.get('screening', {}).items():
    for i, item in enumerate(items):
        if isinstance(item, dict):
            for field in ('stimulus','text','prompt','answer','question'):
                if isinstance(item.get(field), str): add_target(item[field], f'screening.{section}[{i}].{field}')
            for j, option in enumerate(item.get('options', [])):
                add_target(option, f'screening.{section}[{i}].options[{j}]')
        elif isinstance(item, str): add_target(item, f'screening.{section}[{i}]')
for module_id, module in app_data.get('modules', {}).items():
    if str(module_id) == '1': continue  # environmental sound effects are not speech transcripts
    for i, item in enumerate(module.get('items', [])):
        if isinstance(item, str): add_target(item, f'module {module_id} item {i+1}')
        elif isinstance(item, (list, tuple)):
            for j, part in enumerate(item[:2]):
                if isinstance(part, str): add_target(part, f'module {module_id} item {i+1} part {j+1}')
        elif isinstance(item, dict):
            for field in ('stimulus','text','prompt','answer','question'):
                if isinstance(item.get(field), str): add_target(item[field], f'module {module_id} item {i+1} {field}')
            for j, option in enumerate(item.get('options', [])):
                add_target(option, f'module {module_id} item {i+1} option {j+1}')

EXTRA_TARGETS = []  # Add exact Hindi utterances here if needed.
for s in EXTRA_TARGETS: add_target(s, 'manual target')
print('Unique target strings:', len(targets))
pd.DataFrame([{'target':v['text'],'app_locations':'; '.join(v['origins'])} for v in targets.values()]).head(30)


## 2. Search female-speaker transcripts in streaming mode
This avoids intentionally saving the full dataset locally, but the dataset contains 11,825 rows and is stored in large Parquet shards. Streaming can still require substantial network transfer and time to inspect rows. Stop if the runtime begins downloading large amounts of data or if this is not acceptable for your connection.

The dataset card defines gender label `0` as female and `1` as male. This cell records only exact full-transcript matches; it does not use approximate matches as clinical stimuli.

In [ ]:
from datasets import load_dataset

DATASET_ID = 'SPRINGLab/IndicTTS-Hindi'
matches = {}
target_keys = set(targets.keys())
print('Starting streaming transcript scan. This may take time.')
ds = load_dataset(DATASET_ID, split='train', streaming=True)

for idx, row in enumerate(ds):
    text = row.get('text', '')
    gender = row.get('gender', None)
    # HF class label mapping in the dataset card: 0=female, 1=male.
    is_female = (gender == 0 or str(gender).strip().lower() == 'female')
    if is_female:
        key = normalize_text(text)
        if key in target_keys and key not in matches:
            matches[key] = {'text': text, 'row': row, 'dataset_row_index': idx}
            print(f'Exact female match {len(matches)}: {text}')
    if (idx + 1) % 1000 == 0:
        print(f'Checked {idx+1} rows; exact female matches so far: {len(matches)}')
    if len(matches) == len(target_keys):
        print('All target strings found; stopping early.')
        break

print('Scan finished. Exact female matches:', len(matches), 'of', len(target_keys))


## 3. Review the exact-match report
Only full-utterance matches are eligible for export. Words not found as independent utterances remain unmatched; do not cut them out of a longer sentence unless a speech-language/hearing clinician reviews the segmentation and context.

In [ ]:
report_rows = []
for key, info in targets.items():
    found = matches.get(key)
    report_rows.append({
        'target': info['text'],
        'app_locations': '; '.join(info['origins']),
        'exact_female_match': bool(found),
        'dataset_transcript': found['text'] if found else '',
        'dataset_row_index_during_scan': found['dataset_row_index'] if found else ''
    })
report = pd.DataFrame(report_rows)
report.to_csv('/content/springlab_exact_match_report.csv', index=False, encoding='utf-8-sig')
display(report[report['exact_female_match']].head(100))
print('Report saved to /content/springlab_exact_match_report.csv')


## 4. Export matched female audio clips
Run this only after reviewing the exact-match report and confirming that your intended use is permitted by the IIT Madras IndicTTS licence. Export is local to this Colab session and does not upload to GitHub.

In [ ]:
EXPORT_AUDIO = False  # Change to True only after checking the licence and reviewing matches.
OUT = Path('/content/springlab_female_exact_matches')
OUT.mkdir(parents=True, exist_ok=True)

if not EXPORT_AUDIO:
    print('Audio export is disabled. Verify licensing, then set EXPORT_AUDIO = True and rerun this cell.')
else:
    manifest = []
    for key, info in matches.items():
        row = info['row']
        audio = row.get('audio')
        if not audio:
            print('No audio payload for:', info['text'])
            continue
        # datasets Audio feature may decode to {'array','sampling_rate'}; older versions may expose path/bytes.
        audio_data = audio.get('array') if isinstance(audio, dict) else None
        sr = audio.get('sampling_rate', 48000) if isinstance(audio, dict) else 48000
        if audio_data is None:
            print('Could not decode audio array for:', info['text'], '— review the dataset Audio feature version.')
            continue
        import soundfile as sf
        filename = hashlib.sha256(info['text'].encode('utf-8')).hexdigest()[:16] + '.wav'
        sf.write(str(OUT / filename), audio_data, sr, subtype='PCM_16')
        manifest.append({'text': info['text'], 'file': filename, 'sample_rate': sr, 'speaker': 'female', 'dataset': DATASET_ID})
    (OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Exported clips:', len(manifest), 'to', OUT)

print('Remember: do not publish extracted audio until licence permissions are confirmed.')


## Licence status and citation

The Hugging Face dataset card explicitly says this dataset is subject to the original IndicTTS licence and asks users to read and agree to it. It does not grant a standalone CC BY or CC0 licence. The original IIT Madras database page says requesting/downloading data confirms agreement to the licence. Therefore, treat public redistribution of extracted audio in a GitHub Pages app as **not yet cleared** until the licence is reviewed or written permission is obtained from the Speech Technology Consortium/IIT Madras. Contact listed by the dataset: `smtiitm@gmail.com`.

Citation: Speech Technology Consortium and Hema A. Murthy & S. Umesh (2023), *Indic TTS: A Text-to-Speech Database for Indian Languages*, IIT Madras. https://www.iitm.ac.in/donlab/indictts/
